# 07 — Optional anomaly detection extension

**Estimated time:** 75–100 minutes  
**Prerequisites:** notebooks 00–03; distance, density, and ranking metrics.  
**Depends on:** the same leakage-safe feature representation. **Optional module.**

## Learning objectives

- Define an anomaly operationally rather than equating it with the rare target.
- Compare Isolation Forest and novelty-mode Local Outlier Factor (LOF).
- Evaluate ranked alerts with controlled contamination and precision@k.
- Explain why anomalies, minority classes, noise, and fraud are different concepts.


In [ ]:
import hashlib, json, os, platform, random, warnings
from pathlib import Path
ROOT = Path.cwd()
if not (ROOT / "data").exists() and (ROOT.parent / "data").exists():
    ROOT = ROOT.parent
# Known interoperability/UI warnings do not affect predictions or notebook execution.
warnings.filterwarnings("ignore", message="X does not have valid feature names, but LGBMClassifier")
warnings.filterwarnings("ignore", message="IProgress not found.*")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (average_precision_score, confusion_matrix, log_loss,
                             precision_score, recall_score, roc_auc_score)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

SEED = 42
TARGET = "y"
LEAKAGE_COLUMNS = ["duration"]

def project_root():
    '''Return the course root when present, otherwise the notebook directory.'''
    # Return the course root when present, otherwise the notebook's directory.
    return ROOT

def set_seed(seed=SEED):
    '''Seed Python and NumPy RNGs for reproducible notebook runs.'''
    random.seed(seed)
    np.random.seed(seed)

def fast_mode():
    '''Report whether notebooks should use the reduced fast-mode sample.'''
    # Set FAST_MODE=0 for full-size experiments; laptop mode is the default.
    return os.getenv("FAST_MODE", "1").lower() not in {"0", "false", "no"}

def bank_data_path():
    '''Locate the bundled Bank Marketing CSV file.'''
    # The course ships with a local dataset; notebooks never access the network.
    path = project_root() / "data" / "raw" / "bank-full.csv"
    if not path.is_file():
        raise FileNotFoundError(
            f"Expected the bundled Bank Marketing data at {path}. "
            "Run the notebook from the course root or place bank-full.csv there."
        )
    return path

def file_sha256(path):
    '''Compute the SHA-256 digest of a local file.'''
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()

def load_bank_data(include_duration=False):
    '''Load the Bank Marketing dataset and drop leakage columns by default.'''
    # Load the data, encode y, and exclude post-call duration by default.
    frame = pd.read_csv(bank_data_path(), sep=";")
    frame[TARGET] = frame[TARGET].map({"no": 0, "yes": 1}).astype("int8")
    if not include_duration:
        frame = frame.drop(columns=LEAKAGE_COLUMNS)
    return frame

def stratified_sample(frame, n, seed=SEED):
    '''Draw a label-preserving sample from a classified dataset.'''
    if n >= len(frame):
        return frame.copy()
    fractions = frame[TARGET].value_counts(normalize=True)
    counts = (fractions * n).round().astype(int)
    counts.iloc[0] += n - counts.sum()
    parts = [group.sample(n=min(counts.loc[label], len(group)),
                          random_state=seed + int(label))
             for label, group in frame.groupby(TARGET)]
    return pd.concat(parts).sample(frac=1, random_state=seed).reset_index(drop=True)

def make_splits(frame=None, reduced=None):
    '''Create deterministic stratified train, validation, and test splits.'''
    # Deterministic stratified 60/20/20 split; test stays sealed until notebook 09.
    from sklearn.model_selection import train_test_split
    frame = load_bank_data() if frame is None else frame
    train_val, test = train_test_split(
        frame, test_size=0.20, stratify=frame[TARGET], random_state=SEED)
    train, validation = train_test_split(
        train_val, test_size=0.25, stratify=train_val[TARGET], random_state=SEED)
    reduced = fast_mode() if reduced is None else reduced
    if reduced:
        train = stratified_sample(train, 12_000)
        validation = stratified_sample(validation, 4_000, SEED + 1)
        test = stratified_sample(test, 4_000, SEED + 2)
    return tuple(part.reset_index(drop=True) for part in (train, validation, test))

def split_xy(frame):
    '''Separate a frame into feature matrix and target vector.'''
    return frame.drop(columns=TARGET), frame[TARGET]

def feature_groups(frame):
    '''Identify numeric and categorical feature columns.'''
    features = frame.drop(columns=[TARGET], errors="ignore")
    categorical = features.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
    numerical = features.select_dtypes(include=np.number).columns.tolist()
    return numerical, categorical

def make_preprocessor(frame, scale_numeric=True):
    '''Build the preprocessing pipeline for numeric and categorical features.'''
    # Preprocessing is fitted only inside the enclosing training/CV pipeline.
    numerical, categorical = feature_groups(frame)
    numeric_steps = [("impute", SimpleImputer(strategy="median"))]
    if scale_numeric:
        numeric_steps.append(("scale", StandardScaler()))
    categorical_pipe = Pipeline([
        ("impute", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="infrequent_if_exist",
                                 min_frequency=10, sparse_output=True)),
    ])
    return ColumnTransformer([
        ("numeric", Pipeline(numeric_steps), numerical),
        ("categorical", categorical_pipe, categorical),
    ], sparse_threshold=0.3)

def classification_metrics(y_true, probability, threshold=0.5):
    '''Compute ranking and threshold-based classification metrics.'''
    prediction = np.asarray(probability) >= threshold
    tn, fp, fn, tp = confusion_matrix(y_true, prediction, labels=[0, 1]).ravel()
    return {"roc_auc": roc_auc_score(y_true, probability),
            "pr_auc": average_precision_score(y_true, probability),
            "log_loss": log_loss(y_true, probability),
            "precision": precision_score(y_true, prediction, zero_division=0),
            "recall": recall_score(y_true, prediction, zero_division=0),
            "specificity": tn / (tn + fp) if (tn + fp) else np.nan,
            "cost": float(fp + 5 * fn)}

def threshold_table(y_true, probability, thresholds=None):
    '''Evaluate classification metrics across a list of decision thresholds.'''
    thresholds = np.linspace(0.05, 0.80, 76) if thresholds is None else thresholds
    return pd.DataFrame([{"threshold": float(t),
                          **classification_metrics(y_true, probability, float(t))}
                         for t in thresholds])

def add_domain_features(frame):
    '''Create domain-inspired features for the Bank Marketing dataset.'''
    result = frame.copy()
    result["was_previously_contacted"] = (result["pdays"] != -1).astype("int8")
    result["pdays_clean"] = result["pdays"].replace(-1, np.nan)
    result["contact_pressure"] = result["campaign"] / (1 + result["previous"])
    result["balance_per_age"] = result["balance"] / result["age"].clip(lower=18)
    result["age_band"] = pd.cut(result["age"], bins=[0, 29, 39, 49, 59, np.inf],
                                labels=["<30", "30s", "40s", "50s", "60+"]).astype("object")
    return result.drop(columns=["pdays"])

def environment_metadata():
    '''Collect runtime metadata for experiment tracking.'''
    import sklearn
    return {"python": platform.python_version(), "platform": platform.platform(),
            "numpy": np.__version__, "pandas": pd.__version__,
            "scikit_learn": sklearn.__version__}

def write_json(data, path):
    '''Serialize structured data to a JSON file on disk.'''
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(data, indent=2, sort_keys=True), encoding="utf-8")

set_seed(SEED)
FAST_MODE = fast_mode()
CV_FOLDS = 3 if FAST_MODE else 5
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 30)
print({"FAST_MODE": FAST_MODE, "CV_FOLDS": CV_FOLDS, "seed": SEED})


## A deliberately limited connection

Bank Marketing has no verified anomaly or fraud label. Here an **anomaly** means a row that is
unusual under the historical feature distribution and deserves data-quality or operations review.
It does not mean subscriber, fraudster, or error. To obtain known evaluation labels, we inject a
small set of transparent constraint-breaking records into a copy of validation data. This tests
detection of those perturbations only; it does not validate real-world anomaly discovery.


In [ ]:
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.preprocessing import MaxAbsScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import average_precision_score

development, validation, _sealed_test = make_splits(load_bank_data(), reduced=FAST_MODE)
X_dev, _ = split_xy(development)
X_val, _ = split_xy(validation)
preprocessor = make_preprocessor(development, scale_numeric=True)
Z_dev = preprocessor.fit_transform(X_dev)


In [ ]:
rng = np.random.default_rng(SEED)
contaminated = X_val.copy().reset_index(drop=True)
anomaly_label = np.zeros(len(contaminated), dtype=int)
injected_idx = rng.choice(len(contaminated), size=max(40, len(contaminated) // 50), replace=False)
anomaly_label[injected_idx] = 1
# Controlled data-integrity/operational violations, chosen to be explicit and reproducible.
third = len(injected_idx) // 3
contaminated.loc[injected_idx[:third], "age"] = 120
contaminated.loc[injected_idx[third:2*third], "campaign"] = 250
contaminated.loc[injected_idx[2*third:], "balance"] = 1_000_000
Z_eval = preprocessor.transform(contaminated)
print("injected prevalence:", anomaly_label.mean())


Isolation Forest isolates observations using random partitions; fewer splits imply greater
abnormality. LOF compares local density with neighbors. `novelty=True` is essential when scoring
new rows: default LOF is intended for outlier detection on its training set and exposes a different
prediction interface. Both are sensitive to representation and hyperparameters.


In [ ]:
detectors = {
    "IsolationForest": IsolationForest(
        n_estimators=200 if FAST_MODE else 500, contamination="auto",
        random_state=SEED, n_jobs=-1,
    ),
    "LOF_novelty": LocalOutlierFactor(n_neighbors=35, novelty=True, contamination="auto", n_jobs=-1),
}
scores = {}
for name, detector in detectors.items():
    detector.fit(Z_dev)
    scores[name] = -detector.decision_function(Z_eval)  # larger = more anomalous
score_frame = pd.DataFrame(scores)


In [ ]:
def precision_at_k(labels, score, k):
    order = np.argsort(score)[::-1][:k]
    return float(np.mean(np.asarray(labels)[order]))

k = int(anomaly_label.sum())
evaluation = pd.DataFrame({name: {
    "average_precision": average_precision_score(anomaly_label, score),
    "precision_at_k": precision_at_k(anomaly_label, score, k),
    "k": k,
} for name, score in scores.items()}).T
display(evaluation)

plot_data = score_frame.assign(injected=np.where(anomaly_label == 1, "injected", "original"))
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, name in zip(axes, scores):
    sns.histplot(data=plot_data, x=name, hue="injected", bins=40,
                 stat="density", common_norm=False, element="step", ax=ax)
plt.tight_layout()


Precision@k matches a fixed investigation budget. Average precision measures ranking across all
cutoffs but inherits the synthetic-label definition. Original rows scoring highly may be valid
rare customers, genuine data errors, distribution tails, or artifacts of encoding. They require
review—not automatic deletion.

## Common mistakes and leakage warnings

- Calling the positive supervised class an anomaly merely because it is rare.
- Evaluating on the same rows used to fit an outlier detector.
- Using LOF's training-outlier mode to score future observations.
- Deleting anomalies before understanding whether they are valid business cases.
- Claiming fraud detection without a fraud definition or outcome labels.

## Exercises

1. Inject subtler multivariate anomalies that preserve all univariate ranges.
2. Compare precision@k across neighborhood sizes and random seeds.
3. **Challenge:** design a human-review evaluation for naturally high-scoring rows, including a
   sampling plan that avoids verification bias.

## Summary

Anomaly detection is optional because this dataset supplies no natural anomaly label. Controlled
contamination makes the evaluation honest about what it measures. Ranked alerts can support review,
but anomaly score, rare class, noise, and fraud are not interchangeable.

## References

- [IsolationForest](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.IsolationForest.html)
- [LocalOutlierFactor](https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.LocalOutlierFactor.html)
- [Outlier and novelty detection guide](https://scikit-learn.org/stable/modules/outlier_detection.html)
